# 03 — A deliberately misspecified model

We now change **one fitted signal shape**, while keeping the physical simulator from Notebook 01 fixed. This gives us a controlled disagreement to calibrate. The nominal signal-to-background yield ratio remains $1/100$; the nuisance remains a shape-only scale deformation.

This notebook does three things:

1. Defines the bad model with one mismatch parameter $\epsilon$.
2. Checks normalization and measures the bias using our weighted unbinned Asimov construction.
3. Builds an independent physical-simulator population bank for learning the response in Notebook 04.

We fit full event-level likelihoods throughout. There are no analysis bins or preselection.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

RUN_NAME = "default"  # Use a new name when changing the model or training setup.
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("Run:", RUN)
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from model import load_config, bad_anchors, morph, intensity, asimov_weights, fit
from sampling import sample_process
from utils import HybridModel

config = load_config(RUN)
hybrid = HybridModel.load(RUN)
epsilon = config["epsilon"]       # Change this one number to change the mismatch.
N_POPULATION_PER_PROCESS = 200_000
MU_SCAN = np.linspace(*config["mu_range"], 7)
OUT = RUN / "misspecified"
OUT.mkdir(exist_ok=True)
print(f"Signal admixture epsilon = {epsilon:g}")

## 1. Mix normalized shapes, then interpolate the nuisance

For every scale anchor $j\in\{-,0,+\}$, define

$$
R^{\rm bad}_{S,j}(x)=(1-\epsilon)R^{\rm good}_{S,j}(x)
+\epsilon R^{\rm good}_{B,j}(x),
\qquad R^{\rm bad}_{B,j}(x)=R^{\rm good}_{B,j}(x).
$$

Here $R=p/p_{\rm ref}$, and the **complete** process/reference anchors were normalized on the flow integration bank in Notebook 02. The signal mixture therefore has unit integral on that bank. The piecewise-linear nuisance interpolation preserves this normalization at every $\alpha\in[-1,1]$. No nuisance-dependent normalizer is needed.

We use $0\le\epsilon\le1$. Setting $\epsilon=0$ removes the deliberate mixture mismatch; it does not remove density-estimation error between the learned model and the physical simulator. Keep $\epsilon<1$ for an identifiable signal-shape study.

The expected yields stay $\nu S$ and $B$. In particular, the background fraction mixed into the signal shape is still multiplied by the signal yield. We do not add it to the fixed background yield.

In [ ]:
reference_good = hybrid.reference_anchors()
reference_bad = bad_anchors(reference_good, epsilon)

normalization_rows = []
for alpha in [-1., -.5, 0., .5, 1.]:
    normalization_rows.append({
        "alpha": alpha,
        "good_signal": morph(reference_good, alpha)[:, 0].mean(),
        "bad_signal": morph(reference_bad, alpha)[:, 0].mean(),
        "background": morph(reference_bad, alpha)[:, 1].mean(),
    })
display(pd.DataFrame(normalization_rows).set_index("alpha"))

manifest = json.loads((RUN / "hybrid" / "manifest.json").read_text())
misspecification = {
    "epsilon": float(epsilon),
    "hybrid_model_id": manifest["model_id"],
    "construction": "signal=(1-epsilon)*signal+epsilon*background at all three anchors",
    "alpha_bounds": config["alpha_bounds"],
}
(RUN / "misspecification.json").write_text(json.dumps(misspecification, indent=2) + "\n")

At the nominal nuisance value, the signal/background shape ratio transforms as

$$\frac{R_S^{\rm bad}}{R_B}=(1-\epsilon)\frac{R_S^{\rm good}}{R_B}+\epsilon.$$

The following plots make the change visible. The histogram uses reference-distribution events, so it visualizes the model disagreement rather than the $1/100$ physics mixture.

In [ ]:
plot_indices = np.linspace(0, len(reference_good)-1, min(100_000, len(reference_good)), dtype=int)
rg, rb = reference_good[plot_indices], reference_bad[plot_indices]
q_good = rg[:, 0, 1] / rg[:, 1, 1]
q_bad = rb[:, 0, 1] / rb[:, 1, 1]
q_axis = np.logspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(q_axis, q_axis, "--", color="gray", label="unchanged")
axes[0].loglog(q_axis, (1-epsilon)*q_axis+epsilon, label=f"epsilon={epsilon:g}")
axes[0].set(xlabel="Good signal/background ratio", ylabel="Bad signal/background ratio")
axes[0].legend()
bins = np.linspace(*np.quantile(np.log10(q_good), [.002, .998]), 70)
axes[1].hist(np.log10(q_good), bins=bins, density=True, histtype="step", label="good")
axes[1].hist(np.log10(q_bad), bins=bins, density=True, histtype="step", label="bad")
axes[1].set(xlabel="log10 signal/background shape ratio", ylabel="Reference-event density")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "mixture_mismatch.png", dpi=150)
plt.show()

## 2. The unbinned likelihood and reference-sampled Asimov construction

For an experiment $D=(\{x_i\}_{i=1}^{N},y)$, the fitted negative log likelihood, up to parameter-independent terms, is

$$
L(\nu,\alpha;D)=\nu S+B
-\sum_{i=1}^{N}\log\left[\nu S R_S^{\rm bad}(x_i,\alpha)
+B R_B^{\rm bad}(x_i,\alpha)\right]
+\frac{(y-\alpha)^2}{2}.
$$

The common reference density cancels from all fits and likelihood ratios. The Poisson rate term remains essential.

Draw $M$ reference points and give point $x_i$ the expected-event weight

$$w_i(\mu)=\frac{\mu S R_S^{\rm good}(x_i,0)+B R_B^{\rm good}(x_i,0)}{M}.$$

Replacing the event sum by $\sum_i w_i\log[\cdots]$ constructs the unbinned Asimov objective. Because the anchors are normalized on this bank, $\sum_iw_i=\mu S+B$. Fitting the good model should recover $(\mu,0)$; fitting the bad model measures the response to the deliberate mismatch.

For toys we use the requested $y\sim\mathcal U(-2,2)$, while retaining the Gaussian constraint in $L$. For the **population minimizer**, averaging this law is equivalent to setting $y=0$, since $E[y]=0$ and the remaining $E[y^2]/2$ is parameter independent. This is a chosen calibration ensemble, not toys drawn from the Gaussian auxiliary model.

The nuisance interpolation has a kink at zero. The direct fit checks both smooth branches, their join, and the outer bounds.

In [ ]:
reference_rows = []
for mu in MU_SCAN:
    weights = asimov_weights(reference_good, mu, 0., config)
    good_fit = fit(reference_good, 0., config, weights=weights)
    bad_fit = fit(reference_bad, 0., config, weights=weights)
    reference_rows.append({
        "mu": mu,
        "sum_weights": weights.sum(),
        "expected_yield": mu*config["signal_yield"]+config["background_yield"],
        "good_nu": good_fit["nu"], "good_alpha": good_fit["alpha"],
        "bad_nu": bad_fit["nu"], "bad_alpha": bad_fit["alpha"],
        "fit_success": good_fit["success"] and bad_fit["success"],
    })
reference_response = pd.DataFrame(reference_rows)
reference_response.to_csv(OUT / "reference_asimov_response.csv", index=False)
display(reference_response)

## 3. A population objective from the physical simulator

The previous check isolates the imposed model change inside the learned density family. Our eventual target is the response to the **physical simulator**, including its residual disagreement with the learned densities.

Generate independent nominal signal and background banks, with no event selection. Their population objective is

$$
\overline L_{\rm sim}(\nu,\alpha;\mu)=\nu S+B
-\frac{\mu S}{M_S}\sum_{i\in S}\log I_{\rm bad}(x_i;\nu,\alpha)
-\frac{B}{M_B}\sum_{i\in B}\log I_{\rm bad}(x_i;\nu,\alpha)
+\frac{\alpha^2}{2}.
$$

Here $I_{\rm bad}=\nu S R_S^{\rm bad}+B R_B^{\rm bad}$. The physical generating nuisance is fixed at $\alpha_{\rm gen}=0$ in this first study. Learning a response at varying generating nuisances would require those values as network inputs as well.

The network in Notebook 04 will minimize this same objective directly; it will **not** train on the fitted values below. We save the per-event anchors so it can reuse this population bank. These Monte Carlo averages have finite-sample error, especially for the large background term; an independent population bank will be used for validation.

In [ ]:
population = {}
for process, seed_offset in [("signal", 3101), ("background", 3102)]:
    rng = np.random.default_rng(config["seed"] + seed_offset)
    x = sample_process(process, N_POPULATION_PER_PROCESS, rng, alpha=0.)
    population[process] = bad_anchors(hybrid.anchors(x), epsilon)
    np.save(OUT / f"response_{process}.npy", population[process])

bank_metadata = {
    "epsilon": float(epsilon), "hybrid_model_id": manifest["model_id"],
    "n_per_process": N_POPULATION_PER_PROCESS,
    "alpha_gen": 0., "seed": config["seed"],
    "seed_offsets": {"signal": 3101, "background": 3102},
    "source": "independent physical simulator, no preselection",
}
(OUT / "response_bank.json").write_text(json.dumps(bank_metadata, indent=2) + "\n")

physical_anchors = np.concatenate([population["signal"], population["background"]])
physical_rows = []
for mu in MU_SCAN:
    weights = np.concatenate([
        np.full(len(population["signal"]), mu*config["signal_yield"]/len(population["signal"])),
        np.full(len(population["background"]), config["background_yield"]/len(population["background"])),
    ])
    result = fit(physical_anchors, 0., config, weights=weights)
    physical_rows.append({"mu": mu, **result})
physical_response = pd.DataFrame(physical_rows)
physical_response.to_csv(OUT / "simulator_population_response.csv", index=False)
display(physical_response)

Compare the two response curves. Their difference combines learned-density error and finite population-bank integration error. It is not evidence, by itself, of failure of the response network: we have not trained that network yet.

A fitted parameter on a boundary means the response is constrained by the chosen fit domain. We report those points explicitly rather than extending the linear nuisance model outside its positive interpolation range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(MU_SCAN, MU_SCAN, "--", color="gray", label="identity")
axes[0].plot(reference_response.mu, reference_response.bad_nu, "o-", label="learned-good Asimov -> bad")
axes[0].plot(physical_response.mu, physical_response.nu, "s-", label="physical simulator -> bad")
axes[0].set(xlabel="Physical signal strength mu", ylabel="Fitted signal strength g(mu)")
axes[1].axhline(0., ls="--", color="gray")
axes[1].plot(reference_response.mu, reference_response.bad_alpha, "o-", label="learned-good Asimov -> bad")
axes[1].plot(physical_response.mu, physical_response.alpha, "s-", label="physical simulator -> bad")
axes[1].set(xlabel="Physical signal strength mu", ylabel="Fitted nuisance a(mu)")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "population_response.png", dpi=150)
plt.show()

at_boundary = (
    np.isclose(physical_response.nu, config["nu_bounds"][0], atol=1e-4)
    | np.isclose(physical_response.nu, config["nu_bounds"][1], atol=1e-4)
    | np.isclose(np.abs(physical_response.alpha), 1., atol=1e-4)
)
print("Physical response fits on a boundary:", list(physical_response.loc[at_boundary, "mu"]))
print("Next: train the response and profiling networks in Notebook 04.")

## What is saved, and what comes next?

`misspecification.json` records $\epsilon$ and the trained-density identity. The `misspecified/` folder contains the diagnostic tables, plots, and physical population-bank anchors.

Notebook 04 replaces repeated population minimization by a response network and repeated per-experiment profiling by conditional/global optimizer networks. This notebook's direct fits provide independent numerical comparisons; the neural losses will use the likelihood itself. If you change $\epsilon$ or retrain the densities, rerun this notebook and then retrain the downstream networks and calibration flows.